# 01 — Préparation du DataFrame existant

## Objectif

Nous **ne relançons pas la collecte France Travail**.

Ce notebook reprend directement :

1. le DataFrame `df` déjà présent dans la session Colab ; ou
2. le fichier `offres_emploi_tech.csv` déjà exporté.

Il effectue ensuite les contrôles et nettoyages nécessaires avant de continuer vers l'analyse exploratoire, le NLP et le Machine Learning.

> Le jeu de données existant contient environ 991 offres après dédoublonnage.


## 1. Importer les bibliothèques

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

print(" Bibliothèques importées")

 Bibliothèques importées


## 2. Récupérer le DataFrame existant dans Jupyter






In [3]:
CHEMIN_CSV = Path("offres_emploi_tech.csv")


def charger_dataframe_existant():
    dataframe_memorise = globals().get("df")

    if isinstance(dataframe_memorise, pd.DataFrame) and not dataframe_memorise.empty:
        print(" DataFrame `df` récupéré depuis la mémoire Jupyter")
        return dataframe_memorise.copy()


    fichiers_possibles = [
        CHEMIN_CSV,
        Path("offres_emploi_tech_clean.csv"),
        Path.cwd() / "offres_emploi_tech.csv",
        Path.cwd() / "offres_emploi_tech_clean.csv",
    ]

    for fichier in fichiers_possibles:
        if fichier.exists():
            print(f" Fichier chargé : {fichier.resolve()}")
            return pd.read_csv(fichier, encoding="utf-8-sig")

    fichiers_locaux = [p.name for p in Path.cwd().iterdir() if p.is_file()]

    raise FileNotFoundError(
        "Le DataFrame `df` n'existe pas dans ce kernel Jupyter et aucun CSV "
        "n'a été trouvé.\n\n"
        f"Dossier actuel : {Path.cwd()}\n"
        f"Fichiers présents : {fichiers_locaux}\n\n"
        "Place `offres_emploi_tech.csv` dans ce dossier ou modifie CHEMIN_CSV "
        "avec le chemin complet de ton fichier."
    )


df = charger_dataframe_existant()

print(f"\nDimensions initiales : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
df.head(3)

 Fichier chargé : /Users/maysaramellak/Desktop/FT/offres_emploi_tech.csv

Dimensions initiales : 991 lignes × 13 colonnes


,id,titre,description,type_contrat,experience,ville,secteur,entreprise,salaire_min,salaire_max,salaire_moy,date_publication,mot_cle_source
0,210KHBP,Data Scientist / Data Engineer H/F (H/F),Qui sommes-nous ?\n\nNous sommes passionnés pa...,CDI,4 An(s),69 - VILLEURBANNE,Programmation informatique,AMILTONE,28000.0,0.0,14000.0,2026-06-26 14:59:25.109000+00:00,data scientist
1,210FJKM,Senior Data Scientist (H/F),Vous êtes la référence data de l'entreprise. V...,CDI,3 An(s),75 - Paris,Services des traiteurs,FOODCHERI,60000.0,0.0,30000.0,2026-06-23 15:16:01.961000+00:00,data scientist
2,210FFXM,Data Scientist (H/F),Data Scientist - Valorisation des données de s...,CDD - 12 Mois,6 Mois,75 - Paris (Dept.),Activités hospitalières,ASSISTANCE PUBLIQUE HOPITAUX DE PARIS,NaN,NaN,NaN,2026-06-23 14:32:39.663000+00:00,data scientist


## 3. Vérifier les colonnes nécessaires

In [5]:
colonnes_obligatoires = {
    "id",
    "titre",
    "description",
    "type_contrat",
    "experience",
    "ville",
    "date_publication",
}

colonnes_manquantes = sorted(colonnes_obligatoires - set(df.columns))

if colonnes_manquantes:
    raise ValueError(
        "Colonnes obligatoires manquantes : "
        + ", ".join(colonnes_manquantes)
    )

print(" Les colonnes indispensables sont présentes")
print(df.columns.tolist())

 Les colonnes indispensables sont présentes
['id', 'titre', 'description', 'type_contrat', 'experience', 'ville', 'secteur', 'entreprise', 'salaire_min', 'salaire_max', 'salaire_moy', 'date_publication', 'mot_cle_source']


## 4. Nettoyage général

Nous allons :

- supprimer les doublons ;
- retirer les offres sans description ;
- convertir les dates ;
- nettoyer les textes vides ;
- préserver uniquement les offres utilisables pour la suite.


In [8]:
print(f"Avant nettoyage : {len(df):,} lignes")

df = df.copy()

# Uniformiser les noms de colonnes.
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Identifiants et descriptions indispensables.
df = df.dropna(subset=["id", "description"])
df["id"] = df["id"].astype(str).str.strip()
df["description"] = df["description"].astype(str).str.strip()
df["titre"] = df["titre"].fillna("Poste non renseigné").astype(str).str.strip()

df = df[
    df["id"].ne("")
    & df["description"].ne("")
    & df["description"].str.lower().ne("nan")
]

df = df.drop_duplicates(subset="id", keep="first").copy()

# Dates.
df["date_publication"] = pd.to_datetime(
    df["date_publication"],
    errors="coerce",
    utc=True,
)

df = df.sort_values("date_publication", ascending=False).reset_index(drop=True)

print(f"Après nettoyage : {len(df):,} offres uniques")
print(f"Descriptions manquantes : {df['description'].isna().sum()}")
print(f"Identifiants dupliqués : {df['id'].duplicated().sum()}")

Avant nettoyage : 991 lignes
Après nettoyage : 991 offres uniques
Descriptions manquantes : 0
Identifiants dupliqués : 0


## 5. Corriger les salaires déjà extraits

L'ancienne fonction d'extraction a parfois interprété un salaire comme :

- minimum : `28 000`
- maximum : `0`
- moyenne : `14 000`

Cela vient du `.0` présent dans certains montants retournés par l'API.

Comme le texte et l'unité d'origine du salaire n'ont pas été conservés, nous ne pouvons pas reconstruire tous les cas avec certitude. Nous appliquons donc une correction prudente :

- si le maximum vaut zéro et le minimum est positif, nous utilisons le minimum ;
- si les deux bornes sont positives, nous calculons leur moyenne ;
- nous conservons ensuite uniquement les valeurs plausibles pour un salaire annuel, entre 20 000 € et 150 000 €.

Les montants mensuels ou horaires impossibles à identifier sont écartés plutôt qu'inventés.


In [11]:
colonnes_salaires = ["salaire_min", "salaire_max", "salaire_moy"]

for colonne in colonnes_salaires:
    if colonne not in df.columns:
        df[colonne] = np.nan

    df[colonne] = pd.to_numeric(df[colonne], errors="coerce")

df["salaire_moy_avant_correction"] = df["salaire_moy"]

minimum = df["salaire_min"]
maximum = df["salaire_max"]
moyenne_originale = df["salaire_moy"]

salaire_corrige = moyenne_originale.copy()

# Deux bornes positives : moyenne normale.
masque_deux_bornes = (minimum > 0) & (maximum > 0)
salaire_corrige.loc[masque_deux_bornes] = (
    minimum.loc[masque_deux_bornes]
    + maximum.loc[masque_deux_bornes]
) / 2

# Cas fréquent causé par le `.0` du montant.
masque_max_zero = (minimum > 0) & (maximum == 0)
salaire_corrige.loc[masque_max_zero] = minimum.loc[masque_max_zero]

masque_min_zero = (minimum == 0) & (maximum > 0)
salaire_corrige.loc[masque_min_zero] = maximum.loc[masque_min_zero]

# Fourchette prudente d'un salaire annuel tech en France.
masque_plausible = salaire_corrige.between(20_000, 150_000)
df["salaire_moy"] = salaire_corrige.where(masque_plausible)

df["salaire_corrige_heuristique"] = (
    df["salaire_moy"].notna()
    & (
        df["salaire_moy_avant_correction"].isna()
        | ~np.isclose(
            df["salaire_moy"],
            df["salaire_moy_avant_correction"],
            equal_nan=True,
        )
    )
)

print(
    "Salaires avant correction : "
    f"{df['salaire_moy_avant_correction'].notna().sum():,}"
)
print(
    "Salaires annuels plausibles conservés : "
    f"{df['salaire_moy'].notna().sum():,}"
)
print(
    "Valeurs réparées par l'heuristique : "
    f"{df['salaire_corrige_heuristique'].sum():,}"
)

df[
    [
        "titre",
        "salaire_min",
        "salaire_max",
        "salaire_moy_avant_correction",
        "salaire_moy",
    ]
].dropna(subset=["salaire_moy"]).head(10)

Salaires avant correction : 254
Salaires annuels plausibles conservés : 199
Valeurs réparées par l'heuristique : 199


,titre,salaire_min,salaire_max,salaire_moy_avant_correction,salaire_moy
1,Data manager (F/H),40000.0,0.0,20000.0,40000.0
3,Consultant BI / Décisionnel (H/F),40000.0,0.0,20000.0,40000.0
4,Chargé de projet data analyst (H/F),35000.0,0.0,17500.0,35000.0
5,CDD - Responsable d'Analyse Master Data Achats...,40000.0,0.0,20000.0,40000.0
6,ECADE-X Data Space Services Solution Owner H/F,50000.0,0.0,25000.0,50000.0
7,Data analyst & development H/F,50000.0,0.0,25000.0,50000.0
8,Business Analyst for Data Compliance H/F,40000.0,0.0,20000.0,40000.0
9,Data engineer (H/F),50000.0,0.0,25000.0,50000.0
10,Hardware Verification Engineer H/F (H/F),40000.0,0.0,20000.0,40000.0
11,Hardware Verification Engineer H/F (H/F),40000.0,0.0,20000.0,40000.0


## 6. Nettoyer la localisation et le contrat

In [13]:
def extraire_departement(localisation):
    if pd.isna(localisation):
        return None

    resultat = re.match(r"^\s*(\d{2,3})\s*-", str(localisation))
    return resultat.group(1) if resultat else None


def nettoyer_ville(localisation):
    if pd.isna(localisation):
        return None

    ville = str(localisation).strip()

    if " - " in ville:
        ville = ville.split(" - ", 1)[1].strip()

    ville = re.sub(
        r"\s+\d+(?:er|e)?\s+Arrondissement$",
        "",
        ville,
        flags=re.IGNORECASE,
    )

    if ville.lower().startswith("paris"):
        return "Paris"
    if ville.lower().startswith("lyon"):
        return "Lyon"
    if ville.lower().startswith("marseille"):
        return "Marseille"

    return ville


def simplifier_contrat(contrat):
    if pd.isna(contrat):
        return "Non renseigné"

    contrat = str(contrat)

    if "CDI" in contrat:
        return "CDI"
    if "CDD" in contrat:
        return "CDD"
    if "Intérim" in contrat:
        return "Intérim"
    if "Alternance" in contrat or "Apprentissage" in contrat:
        return "Alternance"

    return "Autre"


df["departement"] = df["ville"].apply(extraire_departement)
df["ville_clean"] = df["ville"].apply(nettoyer_ville)
df["contrat_simplifie"] = df["type_contrat"].apply(simplifier_contrat)

print(" Localisations et contrats préparés")
df[["ville", "departement", "ville_clean", "type_contrat", "contrat_simplifie"]].head()

 Localisations et contrats préparés


,ville,departement,ville_clean,type_contrat,contrat_simplifie
0,55 - BRAS-SUR-MEUSE,55,BRAS-SUR-MEUSE,CDI,CDI
1,31 - Toulouse,31,Toulouse,CDI,CDI
2,93 - ST DENIS,93,ST DENIS,CDI,CDI
3,84 - Avignon,84,Avignon,CDI,CDI
4,75 - PARIS,75,Paris,CDD - 6 Mois,CDD


## 7. Contrôle final du jeu de données

In [16]:
print("=" * 60)
print("RÉSUMÉ DU DATAFRAME PRÉPARÉ")
print("=" * 60)
print(f"Nombre d'offres             : {len(df):,}")
print(f"Nombre de colonnes          : {df.shape[1]}")
print(f"Offres avec salaire valide  : {df['salaire_moy'].notna().sum():,}")
print(f"Part avec salaire valide    : {df['salaire_moy'].notna().mean() * 100:.1f} %")
print(f"Nombre de villes            : {df['ville_clean'].nunique(dropna=True):,}")
print(f"Nombre de départements      : {df['departement'].nunique(dropna=True):,}")
print()

print("Principaux contrats :")
print(df["contrat_simplifie"].value_counts(dropna=False))
print()

print("Salaires annuels conservés :")
print(df["salaire_moy"].describe().round(0))

RÉSUMÉ DU DATAFRAME PRÉPARÉ
Nombre d'offres             : 991
Nombre de colonnes          : 18
Offres avec salaire valide  : 199
Part avec salaire valide    : 20.1 %
Nombre de villes            : 284
Nombre de départements      : 72

Principaux contrats :
contrat_simplifie
CDI        668
CDD        195
Intérim    115
Autre       13
Name: count, dtype: int64

Salaires annuels conservés :
count       199.0
mean      44400.0
std       12490.0
min       20000.0
25%       36000.0
50%       44000.0
75%       50000.0
max      130000.0
Name: salaire_moy, dtype: float64


## 8. Enregistrer le DataFrame préparé

Deux fichiers sont générés :

- `offres_emploi_tech.csv` pour rester compatible avec les notebooks existants ;
- `offres_emploi_tech_clean.csv` pour identifier clairement la version nettoyée.


In [18]:
df.to_csv(
    "offres_emploi_tech.csv",
    index=False,
    encoding="utf-8-sig",
)

df.to_csv(
    "offres_emploi_tech_clean.csv",
    index=False,
    encoding="utf-8-sig",
)

print(" Fichier compatible sauvegardé : offres_emploi_tech.csv")
print(" Version nettoyée sauvegardée  : offres_emploi_tech_clean.csv")
print(f" {len(df):,} offres prêtes pour l'analyse")

 Fichier compatible sauvegardé : offres_emploi_tech.csv
 Version nettoyée sauvegardée  : offres_emploi_tech_clean.csv
 991 offres prêtes pour l'analyse


## Étape suivante

Le notebook **02 — Analyse exploratoire** pourra maintenant commencer directement avec :

```python
df = pd.read_csv("offres_emploi_tech_clean.csv", encoding="utf-8-sig")
```

Aucune nouvelle collecte via l'API France Travail ne sera nécessaire.
